# CCTV Shoplifting Detection: all-in-one Colab runner

**CSE 6367 project.** 3D-pose-assisted temporal anomaly detection for shoplifting in retail CCTV.

**Aim.** Detect shoplifting from *human pose sequences* instead of raw pixels (privacy preserving). The research question from the mid-project report:
*does lifting 2D retail skeletons to 3D with a pretrained MotionBERT give a better motion representation for shoplifting detection than 2D pose alone?*

```
RetailS 2D pose (COCO-17) -> quality-gated preprocessing -> [MotionBERT 2D->3D lift] -> features -> BiGRU autoencoder (trained on normal only)
   -> reconstruction error = anomaly score -> threshold = p95 of NORMAL validation errors -> "review" flag
```

| Exp | Input | Detector |
|---|---|---|
| A | 2D pose | BiGRU AE |
| B | 2D pose + velocity | BiGRU AE |
| C | MotionBERT 3D pose | BiGRU AE |
| D | 3D pose + velocity + geometry | BiGRU AE |
| E | D features | BiGRU AE + temporal attention |

**What this notebook does, top to bottom** (it only calls the repo's own `scripts/` and `src/`; no pipeline logic is re-implemented here):
1. Clones the GitHub repo and installs dependencies.
2. Runs the unit-test suite.
3. Downloads RetailS (official authors' Google Drive link, via `gdown`; no Kaggle account needed) and verifies it.
4. Downloads MotionBERT code + the H36M-finetuned checkpoint (Hugging Face, SHA-256 verified) and validates the 2D->3D lift.
5. Trains and evaluates the matched A-E experiments (the `research_v1` protocol).
6. Shows a results table, learning curves, score distributions and ROC curves, then runs PASS/FAIL checks.
7. Optional: expanded 3-seed run, frame-level diagnostic.
7c. Harsha's supervised MNNIT clip baseline (frozen MobileNetV3-small + logistic head; separate real staged-clip dataset, not RetailS).
7d. Improved MNNIT classifier v2 (letterboxed frames, EfficientNet-B0 + R(2+1)D video backbone, 5-fold CV, recall-tuned threshold).
10. **Manual test:** upload any video (any resolution / aspect ratio) and get shoplifting yes/no with a score.
8. Runs the trained detector on **your own CCTV video** (YOLO Pose + ByteTrack -> lift -> score) and plays the annotated video inline.
9. Zips the results for download.

**Runtime:** use a **T4 GPU** runtime (Runtime -> Change runtime type) for the v2 video backbone; the RetailS research scripts run on CPU either way. The matched A-E stage takes roughly 30-60 min on a standard Colab CPU runtime; the first data download is ~1 GB.
Turn on `USE_DRIVE` to cache the dataset, weights and trained models in Google Drive so a disconnect or a second session does not redo the work.

## 0. Settings

In [ ]:
REPO_URL = "https://github.com/hrshimpi/cctv-shoplifting-detection.git"
BRANCH = "claude/project-thread-l20a6j"   # has the v2 classifier + manual test; switch to "main" once the PR is merged

USE_DRIVE = True        # cache RetailS.zip, MotionBERT weights and outputs/ in Google Drive (recommended)
DRIVE_DIR = "/content/drive/MyDrive/cctv-shoplifting-detection-colab"

RUN_TESTS = True        # unit tests (~10 s)
RUN_INPUT_VALIDATION = True   # MotionBERT lift sanity plots (~2-5 min)
RUN_MATCHED = True      # Experiments A-E, matched protocol (main result)
RUN_EXPANDED = False    # Expanded A: 100 train files x 3 seeds. Slow on CPU (hours); off by default
RUN_FRAME_DIAGNOSTIC = True   # secondary frame-level metrics (needs matched + expanded; skipped if missing)
RUN_MNNIT = True        # Harsha's supervised clip baseline on the MNNIT V1 dataset (section 7c, ~760 MB download)
RUN_MNNIT_V2 = True     # improved clip classifier (section 7d); needs RUN_MNNIT's prepared data
TARGET_RECALL = 0.85    # v2 threshold is set on cross-validation scores to catch at least this share of shoplifting clips
MANUAL_TEST = True      # section 10: upload one video and get shoplifting yes/no
RUN_VIDEO_DEMO = False  # score your own video with the trained model (set VIDEO_PATH, or True to be asked for an upload)
VIDEO_PATH = ""         # optional: path to an .mp4 (e.g. on Drive). Empty = upload in section 8
VIDEO_RUN = "matched_d_seed0"   # model used for the video demo (predeclared default in the repo)

## 1. Environment, Drive and repository

In [ ]:
import os, sys, shutil, subprocess, json, hashlib
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = Path("/content/cctv-shoplifting-detection")
else:                                  # local / test execution: run inside an existing checkout
    REPO = Path(os.environ.get("SHOPLIFT_REPO", Path.cwd())).resolve()
    USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path(DRIVE_DIR); DRIVE.mkdir(parents=True, exist_ok=True)
else:
    DRIVE = None

def sh(cmd, check=True):
    # Run a shell command from the repo root and stream its stdout+stderr into this cell
    # (plain subprocess.run output does not show up in Colab, which hides why a command failed).
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in proc.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line])[-40:]
    code = proc.wait()
    if check and code:
        raise RuntimeError(f"Command failed ({code}): {cmd}\n--- last output ---\n" + "".join(tail))
    return code

if IN_COLAB:
    if not (REPO / ".git").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
os.chdir(REPO); sys.path.insert(0, str(REPO))
sh("git log --oneline -3", check=False)
print("Python", sys.version.split()[0], "| repo:", REPO)

In [ ]:
# Keep outputs/ (checkpoints, logs, caches, videos) on Drive so reruns reuse verified work.
OUT = REPO / "outputs"
if DRIVE is not None:
    persisted = DRIVE / "outputs"
    persisted.mkdir(exist_ok=True)
    if not OUT.is_symlink():
        if OUT.exists():
            for item in OUT.iterdir():           # keep any files already produced
                target = persisted / item.name
                if not target.exists():
                    shutil.move(str(item), str(target))
            shutil.rmtree(OUT)
        OUT.symlink_to(persisted, target_is_directory=True)
    print("outputs/ ->", OUT.resolve())
else:
    OUT.mkdir(exist_ok=True)

## 2. Install dependencies
Keeps Colab's preinstalled PyTorch. `imageio-ffmpeg` is needed for the H.264 video export but is missing from `requirements.txt`, so it is added here.

In [ ]:
sh(f"{sys.executable} -m pip install -q -r requirements.txt -r requirements-data.txt imageio-ffmpeg pytest")
import torch, numpy as np
print("torch", torch.__version__, "| numpy", np.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Unit tests

In [ ]:
if RUN_TESTS:
    sh(f"{sys.executable} -m pytest -q")

## 4. RetailS dataset

Source: [TeCSAR-UNCC/RetailS](https://github.com/TeCSAR-UNCC/RetailS) (pose-only, 6 cameras, COCO-17 keypoints). `scripts/prepare_retails.py` downloads the authors' public Google Drive archive with `gdown`, extracts it safely and runs `scripts/verify_retails.py`.

If Google Drive reports a quota / "too many users" error: open the link from the RetailS README in your browser, save `RetailS.zip` into your Drive folder (`DRIVE_DIR` above), and rerun this cell.

In [ ]:
from src.data.retails import discover_retails_root
data_dir = REPO / "data/retails"; data_dir.mkdir(parents=True, exist_ok=True)
archive = data_dir / "RetailS.zip"
drive_zip = DRIVE / "RetailS.zip" if DRIVE is not None else None

if discover_retails_root(data_dir) is None:
    if not archive.exists() and drive_zip is not None and drive_zip.exists():
        print("Copying cached RetailS.zip from Drive..."); shutil.copy(drive_zip, archive)
    try:
        sh(f"{sys.executable} scripts/prepare_retails.py")      # download (if needed) + extract + verify
    except RuntimeError:
        raise SystemExit("RetailS download failed. Put RetailS.zip (from "
                         "https://github.com/TeCSAR-UNCC/RetailS#dataset-download-instructions) at "
                         f"{drive_zip or archive} and rerun this cell.")
    if drive_zip is not None and archive.exists() and not drive_zip.exists():
        print("Caching RetailS.zip on Drive for next time..."); shutil.copy(archive, drive_zip)
else:
    sh(f"{sys.executable} scripts/verify_retails.py")
print("RetailS root:", discover_retails_root(data_dir))

In [ ]:
# Quick look at the data: files per subset and frame-label balance in the test masks.
v = json.loads((OUT / "logs/retails_verification.json").read_text())
import pandas as pd
rows = {k: {kk: vv for kk, vv in s.items() if not isinstance(vv, (dict, list))}
        for k, s in v["subsets"].items()}
display(pd.DataFrame(rows).T)
print("Note: test_staged is almost entirely anomalous frames, so it has (almost) no normal windows to measure false positives on.")

## 5. MotionBERT (frozen 2D->3D lifter)

Clones the official code and downloads the author's H36M-finetuned **pose** checkpoint from Hugging Face with a pinned revision and SHA-256 check (`scripts/prepare_motionbert.py`). The checkpoint is cached on Drive.

In [ ]:
MB = REPO / "third_party/MotionBERT"
CKPT = MB / "checkpoint/pose3d/MB_ft_h36m/best_epoch.bin"
KNOWN_SHA256 = "597c708bd585646cc6961df057ef73634edb83a4d5c1155d98a4cc0c04e81b98"   # value verified in this project
if not (MB / "lib/model/DSTformer.py").exists():
    sh(f"git clone --depth 1 https://github.com/Walter0807/MotionBERT.git {MB}")
drive_ckpt = DRIVE / "motionbert_best_epoch.bin" if DRIVE is not None else None
if not CKPT.exists() and drive_ckpt is not None and drive_ckpt.exists():
    CKPT.parent.mkdir(parents=True, exist_ok=True); shutil.copy(drive_ckpt, CKPT)

def file_sha256(path):
    with open(path, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()

if sh(f"{sys.executable} scripts/prepare_motionbert.py", check=False):
    # Hugging Face unreachable: accept an existing checkpoint only if it matches the known hash.
    if not (CKPT.exists() and file_sha256(CKPT) == KNOWN_SHA256):
        raise SystemExit("MotionBERT checkpoint download failed and no verified copy is available.")
    print("Using existing verified checkpoint (Hugging Face check skipped).")
if drive_ckpt is not None and not drive_ckpt.exists():
    shutil.copy(CKPT, drive_ckpt)
print("MotionBERT checkpoint OK:", CKPT, round(CKPT.stat().st_size / 2**20, 1), "MiB")

In [ ]:
# Structural validation of the joint mapping + real frozen lift, with skeleton previews.
if RUN_INPUT_VALIDATION:
    sh(f"{sys.executable} scripts/validate_research_inputs.py")
    from IPython.display import Image, display
    report = json.loads((OUT / "logs/research_input_validation.json").read_text())
    print(json.dumps({k: v for k, v in report.items() if not isinstance(v, (list, dict))}, indent=2)[:2000])
    for png in sorted((OUT / "plots/research_input_validation").glob("*.png"))[:4]:
        print(png.name); display(Image(filename=str(png), width=900))

## 6. Train and evaluate the matched A-E experiments

`scripts/run_research_phase.py --stage matched`: same 10 sampled normal training files, same source-video-grouped split, same BiGRU size and 15-epoch budget for all five models. Thresholds are the p95 of **normal validation** errors; test labels never select thresholds or epochs. Completed runs are verified and reused on rerun (if `outputs/` is on Drive).

In [ ]:
if RUN_MATCHED:
    sh(f"{sys.executable} -u scripts/run_research_phase.py --stage matched")

## 7. Results

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
LOGS = OUT / "logs/research_v1"
NAMES = {"a": "A: 2D", "b": "B: 2D+vel", "c": "C: 3D", "d": "D: 3D+vel+geom", "e": "E: D+attention"}

def load_runs(prefix):
    runs = {}
    for folder in sorted(LOGS.glob(f"{prefix}*")):
        if (folder / "evaluation.json").exists():
            runs[folder.name] = json.loads((folder / "evaluation.json").read_text())
    return runs

matched = load_runs("matched_")
rows = []
for name, r in matched.items():
    for subset in ("combined", "test_realworld", "test_staged"):
        m = r["metrics"][subset]
        rows.append(dict(run=NAMES.get(r["experiment"], name), subset=subset, best_epoch=r["best_epoch"],
                         threshold=m.get("threshold"), precision=m["precision"], recall=m["recall"], f1=m["f1"],
                         specificity=m.get("specificity"), fpr=m.get("false_positive_rate"),
                         roc_auc=m.get("roc_auc"), pr_auc=m.get("pr_auc"),
                         normal=m.get("num_normal"), anomaly=m.get("num_anomaly")))
results = pd.DataFrame(rows)
pd.set_option("display.precision", 3)
display(results[results.subset == "combined"].drop(columns="subset").set_index("run"))
display(results[results.subset != "combined"].set_index(["run", "subset"]))

In [ ]:
# Learning curves (normal train / normal validation reconstruction loss)
fig, axes = plt.subplots(1, len(matched), figsize=(4 * len(matched), 3), squeeze=False)
for ax, (name, r) in zip(axes[0], matched.items()):
    h = pd.read_csv(LOGS / name / "training_log.csv")
    ax.plot(h.epoch, h.train_loss, label="train"); ax.plot(h.epoch, h.validation_loss, label="validation")
    ax.axvline(r["best_epoch"], ls="--", c="gray"); ax.set_title(NAMES[r["experiment"]]); ax.set_xlabel("epoch")
axes[0][0].legend(); plt.tight_layout(); plt.show()

# Score distributions with the normal-calibrated threshold, and ROC curves on the combined test set
fig, axes = plt.subplots(1, len(matched) + 1, figsize=(4 * (len(matched) + 1), 3.2))
for ax, (name, r) in zip(axes, matched.items()):
    w = pd.read_csv(LOGS / name / "window_predictions.csv")
    lo, hi = np.percentile(w.score, [0.5, 99.5]); bins = np.linspace(lo, hi, 40)
    ax.hist(w.score[w.label == 0], bins, alpha=.6, density=True, label="normal")
    ax.hist(w.score[w.label == 1], bins, alpha=.6, density=True, label="anomaly")
    ax.axvline(w.threshold.iloc[0], c="red", ls="--", label="threshold"); ax.set_title(NAMES[r["experiment"]])
    fpr, tpr, _ = roc_curve(w.label, w.score)
    axes[-1].plot(fpr, tpr, label=f"{r['experiment'].upper()} AUC={roc_auc_score(w.label, w.score):.3f}")
axes[0].legend(fontsize=7); axes[-1].plot([0, 1], [0, 1], "k:"); axes[-1].set_title("ROC (combined)"); axes[-1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
# Per-camera breakdown for the best 2D (A) and the predeclared 3D model (D)
for name in ("matched_a_seed0", "matched_d_seed0"):
    if not (LOGS / name / "window_predictions.csv").exists():
        continue
    w = pd.read_csv(LOGS / name / "window_predictions.csv")
    cam_col = "camera_id" if "camera_id" in w.columns else None
    if cam_col is None:
        print("No camera column in", name, "- columns:", list(w.columns)); continue
    g = w.groupby(["subset", cam_col]).apply(lambda d: pd.Series(dict(
        windows=len(d), normal=(d.label == 0).sum(), anomaly=(d.label == 1).sum(),
        recall=((d.prediction == 1) & (d.label == 1)).sum() / max(1, (d.label == 1).sum()),
        fpr=((d.prediction == 1) & (d.label == 0)).sum() / max(1, (d.label == 0).sum()),
        roc_auc=roc_auc_score(d.label, d.score) if d.label.nunique() == 2 else np.nan)), include_groups=False)
    print(name); display(g)

### Verification checks
These assert the protocol properties the project claims; every line should say PASS.

In [ ]:
checks = []
def check(name, ok, detail=""):
    checks.append((name, bool(ok))); print(("PASS " if ok else "FAIL ") + name + (f"  ({detail})" if detail else ""))

check("five matched runs evaluated", len(matched) == 5, ", ".join(matched))
windows = {n: pd.read_csv(LOGS / n / "window_predictions.csv") for n in matched}
keys = [tuple(map(tuple, w.drop(columns=[c for c in w.columns if c in ("score", "prediction", "threshold", "outcome")]).values)) for w in windows.values()]
check("all models scored identical test windows and labels", len(set(keys)) == 1)
check("all scores finite", all(np.isfinite(w.score).all() for w in windows.values()))
splits = {json.dumps(r["split"], sort_keys=True) for r in matched.values()}
check("same source-video train/validation split for A-E", len(splits) == 1)
for n, r in matched.items():
    calib = np.load(OUT / "checkpoints/research_v1" / n / "validation_errors.npy")
    check(f"{n}: threshold = p95 of normal validation errors",
          np.isclose(np.percentile(calib, 95), windows[n].threshold.iloc[0]))
    hist = pd.read_csv(LOGS / n / "training_log.csv")
    check(f"{n}: trained 15 epochs, best epoch by validation loss",
          len(hist) == 15 and int(hist.validation_loss.idxmin()) + 1 == r["best_epoch"])
print(f"\n{sum(ok for _, ok in checks)}/{len(checks)} checks passed")

## 7b. Optional: expanded 3-seed run and frame-level diagnostic

`RUN_EXPANDED` trains Experiment A on 100 normal files with seeds 0/1/2 (three CPU worker processes). This takes hours on Colab CPU; the repo's GPU route for it is `notebooks/Retail_Shoplifting_GPU_Training.ipynb` (see `COLAB_TRAINING.md`). The frame diagnostic needs both matched and expanded runs.

In [ ]:
if RUN_EXPANDED:
    sh(f"{sys.executable} -u scripts/run_research_phase.py --stage expanded")
    expanded = load_runs("expanded_a_seed")
    t = pd.DataFrame([{"run": n, **r["metrics"]["combined"]} for n, r in expanded.items()]).set_index("run")
    display(t[["precision", "recall", "f1", "roc_auc", "pr_auc"]]); display(t[["recall", "f1", "roc_auc"]].agg(["mean", "std"]))
if RUN_FRAME_DIAGNOSTIC:
    if all((LOGS / f"expanded_a_seed{s}/evaluation.json").exists() for s in (0, 1, 2)):
        sh(f"{sys.executable} scripts/evaluate_frame_diagnostic.py")
        sh(f"{sys.executable} scripts/summarize_research_phase.py --allow-incomplete")
        print((LOGS / "results.md").read_text()[:4000])
    else:
        print("Frame diagnostic skipped: it needs the expanded A runs (set RUN_EXPANDED = True).")

## 7c. Supervised MNNIT clip baseline (Harsha, commit 041f3dd)

A separate, supervised experiment added by Harsha (`MNNIT_CLIP_CLASSIFICATION.md`). It classifies whole **staged laboratory clips** as normal vs shoplifting:
16 frames per clip -> frozen ImageNet MobileNetV3-small embeddings -> mean / std / mean absolute change (1,728-d) -> StandardScaler + class-balanced logistic regression (C=1, threshold 0.5). Split: duplicate-aware stratified group folds, seed 2026 (train 107 / val 36 / test 36).

Data: Ansari & Singh, *Shoplifting Dataset (2022), CV Lab MNNIT Allahabad*, Mendeley Data V1, DOI 10.17632/r3yjf35hzr.1, CC BY 4.0. The archive is SHA-256 checked by `scripts/prepare_mnnit_clips.py`.

These metrics are **not comparable** with the RetailS A-E results above (different data, labels, split and task), and they say nothing about real-store person/time localization.

If the automatic download fails (Mendeley sometimes blocks scripted requests), open https://data.mendeley.com/datasets/r3yjf35hzr/1 in your browser, click **Download All**, and put the `.zip` in your Drive folder as `DRIVE_DIR/mnnit_Dataset.zip`, then rerun.

In [ ]:
MNNIT_SHA256 = "57657afeff84c44482327c46fdbb1f45b22e9f045bf44cdd31e368b07bebc93c"
mn_zip = REPO / "data/action_sources/mnnit_v1/Dataset.zip"; mn_zip.parent.mkdir(parents=True, exist_ok=True)
drive_mn = DRIVE / "mnnit_Dataset.zip" if DRIVE is not None else None

def find_download_urls(obj, found):
    # Walk the Mendeley public-API JSON and collect (filename, url) pairs for .zip files.
    if isinstance(obj, dict):
        name = obj.get("filename") or obj.get("name") or ""
        url = (obj.get("content_details") or {}).get("download_url") or obj.get("download_url")
        if url and str(name).lower().endswith(".zip"):
            found.append((name, url))
        for v in obj.values(): find_download_urls(v, found)
    elif isinstance(obj, list):
        for v in obj: find_download_urls(v, found)
    return found

if RUN_MNNIT:
    if not mn_zip.exists() and drive_mn is not None and drive_mn.exists():
        print("Copying cached MNNIT archive from Drive..."); shutil.copy(drive_mn, mn_zip)
    if not mn_zip.exists():
        import requests
        headers = {"User-Agent": "Mozilla/5.0 (colab notebook)"}
        try:
            api = "https://data.mendeley.com/public-api/datasets/r3yjf35hzr/files?folder_id=root&version=1"
            urls = find_download_urls(requests.get(api, headers=headers, timeout=60).json(), [])
            print("Mendeley files:", [n for n, _ in urls])
            name, url = urls[0]
            with requests.get(url, headers=headers, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(mn_zip, "wb") as f:
                    for chunk in r.iter_content(1 << 20): f.write(chunk)
        except Exception as error:
            if mn_zip.exists(): mn_zip.unlink()
            raise SystemExit(f"MNNIT download failed ({error!r}). Download it in your browser from "
                             "https://data.mendeley.com/datasets/r3yjf35hzr/1 and save it as "
                             f"{drive_mn or mn_zip}, then rerun this cell.")
    got = file_sha256(mn_zip)
    print("MNNIT archive", round(mn_zip.stat().st_size / 2**20, 1), "MiB, sha256", got[:16], "...", "OK" if got == MNNIT_SHA256 else "MISMATCH")
    if got != MNNIT_SHA256:
        raise SystemExit("Archive does not match the publisher SHA-256 (prepare_mnnit_clips.py would refuse it).")
    if drive_mn is not None and not drive_mn.exists():
        print("Caching MNNIT archive on Drive..."); shutil.copy(mn_zip, drive_mn)
    sh(f"{sys.executable} -u scripts/prepare_mnnit_clips.py")       # verify, extract, decode every frame, dedupe, contact sheets
    sh(f"{sys.executable} -u scripts/train_mnnit_clip_baseline.py")  # frozen split + frozen encoder + two logistic heads
    sh(f"{sys.executable} scripts/build_mnnit_clip_report.py")       # confusion matrices, error sheet, results.md


In [ ]:
from IPython.display import Image, Markdown
if RUN_MNNIT:
    MN = OUT / "mnnit_clip_classification"
    rep = json.loads((MN / "evaluation.json").read_text())
    data_rep = json.loads((MN / "data_report.json").read_text())
    print({k: data_rep[k] for k in ("videos", "canonical_videos", "exact_duplicates_excluded", "near_duplicate_pairs", "decoded_frames", "sizes")})
    print("split counts (label -> clips):", rep["split_counts"])
    t = pd.DataFrame([{"model": m, "split": s, **v} for m, sets in rep["metrics"].items() for s, v in sets.items()])
    display(t[["model", "split", "clips", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "tp", "tn", "fp", "fn"]].set_index(["model", "split"]))
    display(Image(filename=str(MN / "confusion.png"), width=800))
    if (MN / "test_error_examples.jpg").exists():
        display(Image(filename=str(MN / "test_error_examples.jpg"), width=900))
    display(pd.read_csv(MN / "test_errors.csv"))
    # Sanity checks on the protocol Harsha declared
    print(("PASS" if not rep["test_selected_parameters"] else "FAIL"), "no test-selected parameters")
    print(("PASS" if rep["split_counts"]["test"] and sum(rep["split_counts"]["test"].values()) == 36 else "CHECK"), "36 test clips")


## 7d. Improved MNNIT classifier (v2)

What changed vs Harsha's v1, and why:

| v1 problem | v2 change |
|---|---|
| Frames stretched to 320x240 (distorts 16:9 / 9:16 video) | **Letterboxing**: scale to fit, pad with black. Any aspect ratio (4:3, 1:1, 16:9, 9:16) is handled the same way in training and in the manual test |
| 16 frames, ImageNet MobileNetV3-small, "motion" = embedding differences | 32 frames through **EfficientNet-B0**, plus a real **video backbone, R(2+1)D-18 pretrained on Kinetics-400** (4 windows x 16 frames) that sees actual motion |
| One fixed setting, threshold 0.5, validation 94% vs test 69% | **5-fold group cross-validation** on train+val picks the feature set, PCA size and regularisation C; the **threshold is tuned on out-of-fold scores to reach `TARGET_RECALL`** (missing a theft costs more than a false alarm) |

Backbones stay frozen: with ~140 training clips, fine-tuning a video network mostly memorises them, which is v1's overfitting problem made worse.
The test split (36 clips) is the same as v1 and is scored once at the end, with nothing chosen from it. v1 already reported on it, so treat it as a fair comparison, not a pristine hold-out.

In [ ]:
if RUN_MNNIT and RUN_MNNIT_V2:
    sh(f"{sys.executable} -u scripts/train_mnnit_clip_v2.py --target-recall {TARGET_RECALL}")
    V2 = OUT / "mnnit_clip_v2"
    ev = json.loads((V2 / "evaluation.json").read_text())
    print("Backbones:", ev["backbones"], "| selected by CV:", ev["selected"], "| threshold:", round(ev["threshold"], 3))
    display(pd.read_csv(V2 / "cv_results.csv").head(10))
    cols = ["clips", "accuracy", "balanced_accuracy", "precision", "recall", "f1", "roc_auc", "tp", "tn", "fp", "fn"]
    cmp = {"v2 cross-val (train+val, out-of-fold)": ev["cross_validation"]["out_of_fold"],
           "v2 TEST (tuned threshold)": ev["test"], "v2 TEST (threshold 0.5)": ev["test_at_0_5"]}
    if "v1_test_reference" in ev: cmp["v1 TEST (Harsha, threshold 0.5)"] = ev["v1_test_reference"]
    display(pd.DataFrame({k: {c: v.get(c) for c in cols} for k, v in cmp.items()}).T)
    p = pd.read_csv(V2 / "predictions.csv"); t = p[p.split == "test"]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(t.score[t.label == 0], bins=20, range=(0, 1), alpha=.6, label="normal")
    ax.hist(t.score[t.label == 1], bins=20, range=(0, 1), alpha=.6, label="shoplifting")
    ax.axvline(ev["threshold"], c="red", ls="--", label="threshold"); ax.set_title("v2 test scores"); ax.legend(); plt.show()
    display(t[t.label != t.prediction])   # every test error, for inspection


## 8. Run the trained detector on your own CCTV video

Pipeline: YOLOv8 Pose + ByteTrack (`scripts/extract_video_poses.py`) -> same quality preprocessing -> MotionBERT lift -> trained `VIDEO_RUN` model (`scripts/run_video_inference.py`) -> annotated H.264 video + per-window scores.

Red skeleton = window score above the normal-calibrated threshold (flag for human review), green = below, gray = unscored (track too short / poorly visible). Scores are **not** theft probabilities, and processing is offline (whole-track smoothing), not real-time.

In [ ]:
import re
if RUN_VIDEO_DEMO or VIDEO_PATH:
    src = Path(VIDEO_PATH) if VIDEO_PATH else None
    if src is None:
        if not IN_COLAB:
            raise SystemExit("Set VIDEO_PATH to an .mp4 file")
        from google.colab import files
        uploaded = files.upload()                 # choose an .mp4 CCTV clip
        src = Path(next(iter(uploaded)))
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", src.stem).strip("_") or "clip"
    video = REPO / "data/external_video/videos/uploaded" / f"{stem}.mp4"
    video.parent.mkdir(parents=True, exist_ok=True)
    if src.resolve() != video.resolve():
        shutil.copy(src, video)
    pose_dir = OUT / "poses_batch/conf05/uploaded" / stem          # location run_video_inference.py expects
    (REPO / "third_party/yolo").mkdir(parents=True, exist_ok=True)  # Ultralytics downloads yolov8n-pose.pt here
    if (pose_dir / "summary.json").exists() and json.loads((pose_dir / "summary.json").read_text())["source_video"] == str(video.resolve()):
        sh(f'{sys.executable} scripts/extract_video_poses.py --verify-only --out-dir "{pose_dir}"')   # reuse earlier export
    else:
        if pose_dir.exists():
            shutil.rmtree(pose_dir)                # stale export of a different file with the same name
        sh(f'{sys.executable} scripts/extract_video_poses.py --video "{video}" --out-dir "{pose_dir}" --conf-threshold 0.5')
    sh(f'{sys.executable} scripts/run_video_inference.py --run {VIDEO_RUN} --video "{video}"')
    result_dir = OUT / "video_inference" / VIDEO_RUN / stem
    print("Outputs:", result_dir)

In [ ]:
from IPython.display import HTML, display
import base64
if RUN_VIDEO_DEMO or VIDEO_PATH:
    rep = json.loads((result_dir / "scores.json").read_text())
    print({k: rep[k] for k in ("experiment", "threshold", "candidate_windows", "retained_windows", "flagged_windows",
                               "video_frame_score_coverage", "source_fps")})
    w = pd.DataFrame(rep["windows"])
    if len(w):
        fig, ax = plt.subplots(figsize=(10, 3))
        for tid, d in w.groupby("track_id"):
            ax.step(d.end_seconds, d.score, where="post", label=f"track {tid}")
        ax.axhline(rep["threshold"], c="red", ls="--", label="threshold"); ax.set_xlabel("time (s)"); ax.set_ylabel("anomaly score")
        ax.legend(fontsize=7, ncol=4); plt.show()
        display(w.sort_values("score", ascending=False).head(10))
    data = base64.b64encode((result_dir / "annotated.mp4").read_bytes()).decode()
    if len(data) < 60e6:
        display(HTML(f'<video width="800" controls src="data:video/mp4;base64,{data}"></video>'))
    else:
        print("Video too large to embed; download it from", result_dir / "annotated.mp4")

## 9. Download the results

In [ ]:
bundle = Path("/content/shoplifting_results.zip" if IN_COLAB else REPO / "outputs/shoplifting_results.zip")
import zipfile
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for base in [OUT / "logs/research_v1", OUT / "video_inference", OUT / "plots/research_input_validation", OUT / "mnnit_clip_classification", OUT / "mnnit_clip_v2"]:
        for p in base.rglob("*") if base.exists() else []:
            if p.is_file() and p.name not in ("sampled_rgb.npz", "embeddings.npz") and "extracted" not in p.parts and p.suffix in {".json", ".csv", ".md", ".png", ".jpg", ".mp4"} and p.stat().st_size < 200e6:
                z.write(p, p.relative_to(OUT))
    for f in ("logs/retails_verification.json", "logs/motionbert_setup.json", "logs/research_input_validation.json"):
        if (OUT / f).exists(): z.write(OUT / f, f)
print(bundle, round(bundle.stat().st_size / 2**20, 1), "MiB")
if IN_COLAB:
    from google.colab import files; files.download(str(bundle))

## 10. Manual test: upload a video, get shoplifting yes / no

Uses the v2 model from section 7d (`scripts/predict_mnnit_clip.py`). Any resolution and aspect ratio works (4:3, 1:1, 16:9, 9:16...): frames are letterboxed exactly as in training, never stretched.
Videos longer than ~15 s are cut into 10 s chunks; each chunk gets a score and the video is flagged if any chunk crosses the threshold.

Caveats: the model learned from staged lab clips, so treat results on real store footage as a demo, not a verified detector. The score is not a calibrated probability. Rerun this cell to test another video.

In [ ]:
MANUAL_VIDEO_PATH = ""     # optional: a path on Drive, e.g. "/content/drive/MyDrive/test.mp4"; empty = upload
if MANUAL_TEST:
    assert (OUT / "mnnit_clip_v2/model.npz").exists(), "Run section 7d first (RUN_MNNIT and RUN_MNNIT_V2)."
    if MANUAL_VIDEO_PATH:
        test_video = Path(MANUAL_VIDEO_PATH)
    else:
        from google.colab import files
        up = files.upload()                       # pick one video file
        test_video = Path(next(iter(up))).resolve()
    sys.path.insert(0, str(REPO))
    from scripts.predict_mnnit_clip import predict
    r = predict(test_video, chunk_seconds=10)
    verdict = "SHOPLIFTING SUSPECTED" if r["shoplifting"] else "NO SHOPLIFTING DETECTED"
    color = "#c62828" if r["shoplifting"] else "#2e7d32"
    display(HTML(f'<h2 style="color:{color}">{verdict}</h2><p>max score {r["max_score"]:.3f} vs threshold {r["threshold"]:.3f} | '
                 f'{r["width"]}x{r["height"]} ({r["aspect"]}), {r["duration_s"]} s, {r["fps"]} fps</p>'))
    display(pd.DataFrame(r["chunks"]))
    # Show what the model sees: a middle frame before and after letterboxing
    import cv2
    from IPython.display import HTML
    from src.data.video_clip_features import letterbox
    cap = cv2.VideoCapture(str(test_video)); cap.set(cv2.CAP_PROP_POS_FRAMES, r["frames"] // 2); ok, fr = cap.read(); cap.release()
    if ok:
        fr = cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].imshow(fr); ax[0].set_title(f"original {r['width']}x{r['height']}"); ax[1].imshow(letterbox(fr, 224)); ax[1].set_title("model input (letterboxed 224x224)")
        for a in ax: a.axis("off")
        plt.show()
